In [1]:
!unzip mallorn-astronomical-classification-challenge.zip

Archive:  mallorn-astronomical-classification-challenge.zip
  inflating: sample_submission.csv   
  inflating: split_01/test_full_lightcurves.csv  
  inflating: split_01/train_full_lightcurves.csv  
  inflating: split_02/test_full_lightcurves.csv  
  inflating: split_02/train_full_lightcurves.csv  
  inflating: split_03/test_full_lightcurves.csv  
  inflating: split_03/train_full_lightcurves.csv  
  inflating: split_04/test_full_lightcurves.csv  
  inflating: split_04/train_full_lightcurves.csv  
  inflating: split_05/test_full_lightcurves.csv  
  inflating: split_05/train_full_lightcurves.csv  
  inflating: split_06/test_full_lightcurves.csv  
  inflating: split_06/train_full_lightcurves.csv  
  inflating: split_07/test_full_lightcurves.csv  
  inflating: split_07/train_full_lightcurves.csv  
  inflating: split_08/test_full_lightcurves.csv  
  inflating: split_08/train_full_lightcurves.csv  
  inflating: split_09/test_full_lightcurves.csv  
  inflating: split_09/train_full_lightcurves

In [2]:
import pandas as pd
train_log  = pd.read_csv('train_log.csv')
train_log = train_log.drop(
    columns=['Z_err', 'SpecType', 'English Translation']
)


In [3]:
print(train_log.head)

<bound method NDFrame.head of                      object_id       Z    EBV     split  target
0     Dornhoth_fervain_onodrim  3.0490  0.110  split_01       0
1          Dornhoth_galadh_ylf  0.4324  0.058  split_01       0
2         Elrim_melethril_thul  0.4673  0.577  split_01       0
3           Ithil_tobas_rodwen  0.6946  0.012  split_01       0
4          Mirion_adar_Druadan  0.4161  0.058  split_01       0
...                        ...     ...    ...       ...     ...
3038       tinnu_gellui_tathar  0.8898  0.042  split_20       0
3039            uir_heleg_corf  0.9598  0.042  split_20       0
3040             uir_rhosc_law  0.1543  0.024  split_20       0
3041              uruk_in_pess  1.1520  0.019  split_20       0
3042           ylf_alph_mindon  0.5595  0.034  split_20       0

[3043 rows x 5 columns]>


In [4]:
import numpy as np
data_log_np = train_log.to_numpy()

In [5]:
print(data_log_np[0])

['Dornhoth_fervain_onodrim' 3.049 0.11 'split_01' 0]


In [6]:
split_map = {}
splits_names  = {'split_01', 'split_02' , 'split_03' , 'split_04' , 'split_05', 'split_06' , 'split_07' , 'split_08' , 'split_09', 'split_10' , 'split_11' , 'split_12' , 'split_13', 'split_14' , 'split_15' , 'split_16' , 'split_17', 'split_18' , 'split_19' , 'split_20'}


In [7]:
!pip install catboost
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 11.4 MB/s eta 0:00:00


In [8]:
filters = ['u','g','r','i','z','y']
quantiles = [0.25, 0.5, 0.75]
for x in splits_names :
  dta = pd.read_csv(x + '/train_full_lightcurves.csv')
  split_map[x] = dta
import numpy as np
from scipy.stats import skew, kurtosis

from scipy.stats import skew, kurtosis
import numpy as np

filters = ['u','g','r','i','z','y']
quantiles = [0.1,0.25,0.5,0.75,0.9]

N_FEATURES_PER_FILTER = 14 + len(quantiles)

def extract_features(df_obj):
    features = []
    peak_times = {}
    mean_fluxes = {}

    for f in filters:
        df_f = df_obj[df_obj['Filter'] == f]

        if len(df_f) < 3:
            features.extend([0]*N_FEATURES_PER_FILTER)
            peak_times[f] = 0
            mean_fluxes[f] = 0
            continue

        flux = df_f['Flux'].values
        time = df_f['Time (MJD)'].values
        ferr = df_f['Flux_err'].values

        idx_peak = np.argmax(flux)

        amp = flux.max() - flux.min()
        mean_flux = flux.mean()
        std_flux = flux.std()
        median_flux = np.median(flux)
        snr = np.mean(np.abs(flux) / (ferr + 1e-6))

        skew_flux = skew(flux)
        kurt_flux = kurtosis(flux)

        duration = time.max() - time.min()
        n_points = len(flux)

        rise_time = time[idx_peak] - time.min()
        fall_time = time.max() - time[idx_peak]

        slope_rise = (flux[idx_peak] - flux.min()) / (rise_time + 1e-6)
        slope_fall = (flux[idx_peak] - flux[-1]) / (fall_time + 1e-6)

        variability = std_flux / (mean_flux + 1e-6)

        q = np.quantile(flux, quantiles)
        q = list(q)

        feats = [
            amp, mean_flux, std_flux, median_flux, snr,
            skew_flux, kurt_flux,
            duration, n_points,
            rise_time, fall_time,
            slope_rise, slope_fall,
            variability
        ] + q

        features.extend(feats)

        peak_times[f] = time[idx_peak]
        mean_fluxes[f] = mean_flux

    color_pairs = [('g','r'), ('r','i'), ('i','z')]
    for f1, f2 in color_pairs:
        features.append(mean_fluxes.get(f1,0) - mean_fluxes.get(f2,0))

    for f1, f2 in color_pairs:
        features.append(peak_times.get(f1,0) - peak_times.get(f2,0))

    return np.array(features, dtype=float)

X_train = []
y_train = []

for row in data_log_np:
    df_split = split_map[row[3]]
    df_obj = df_split[df_split['object_id']==row[0]]
    feats = extract_features(df_obj)
    X_train.append(feats)
    y_train.append(row[-1])

X_train = np.array(X_train)
y_train = np.array(y_train)


In [9]:
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)


In [10]:

xgb_model = XGBClassifier(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.015,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=10,
    gamma=0.3,
    reg_alpha=0.5,
    reg_lambda=1.5,
    scale_pos_weight=19,
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42
)

cat_model = CatBoostClassifier(
    iterations=2000,
    depth=5,
    learning_rate=0.02,
    loss_function="Logloss",
    eval_metric="F1",
    class_weights=[1,19],
    l2_leaf_reg=7,
    bagging_temperature=0.7,
    border_count=128,
    random_seed=42,
    verbose=200
)


In [11]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X_train_imp))
oof_cat = np.zeros(len(X_train_imp))

for tr_idx, va_idx in skf.split(X_train_imp, y_train):
    X_tr, X_va = X_train_imp[tr_idx], X_train_imp[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]
    xgb_model.fit(X_tr, y_tr)
    oof_xgb[va_idx] = xgb_model.predict_proba(X_va)[:,1]

    cat_model.fit(X_tr, y_tr)
    oof_cat[va_idx] = cat_model.predict_proba(X_va)[:,1]

oof_ens = 0.5*oof_xgb + 0.5*oof_cat

thresholds = np.linspace(0.01,0.2,50)
best_f1 = 0
best_thresh = 0
for t in thresholds:
    preds = (oof_ens > t).astype(int)
    f1 = f1_score(y_train, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best OOF F1:", best_f1)
print("Best threshold:", best_thresh)


0:	learn: 0.8083387	total: 58ms	remaining: 1m 55s
200:	learn: 0.9620253	total: 1.81s	remaining: 16.2s
400:	learn: 0.9829022	total: 3.6s	remaining: 14.4s
600:	learn: 0.9966659	total: 5.32s	remaining: 12.4s
800:	learn: 0.9993314	total: 7.1s	remaining: 10.6s
1000:	learn: 1.0000000	total: 8.83s	remaining: 8.82s
1200:	learn: 1.0000000	total: 12.3s	remaining: 8.2s
1400:	learn: 1.0000000	total: 15.8s	remaining: 6.74s
1600:	learn: 1.0000000	total: 19.5s	remaining: 4.85s
1800:	learn: 1.0000000	total: 21.7s	remaining: 2.4s
1999:	learn: 1.0000000	total: 24.8s	remaining: 0us
0:	learn: 0.7953866	total: 11ms	remaining: 22.1s
200:	learn: 0.9581197	total: 1.77s	remaining: 15.8s
400:	learn: 0.9826868	total: 4.86s	remaining: 19.4s
600:	learn: 0.9962231	total: 6.6s	remaining: 15.4s
800:	learn: 0.9995542	total: 8.34s	remaining: 12.5s
1000:	learn: 1.0000000	total: 10.1s	remaining: 10.1s
1200:	learn: 1.0000000	total: 11.8s	remaining: 7.85s
1400:	learn: 1.0000000	total: 13.5s	remaining: 5.78s
1600:	learn: 1.

In [12]:
xgb_model.fit(X_train_imp, y_train)
cat_model.fit(X_train_imp, y_train)


0:	learn: 0.7987795	total: 23.9ms	remaining: 47.9s
200:	learn: 0.9498396	total: 2s	remaining: 17.9s
400:	learn: 0.9762194	total: 3.79s	remaining: 15.1s
600:	learn: 0.9932886	total: 5.63s	remaining: 13.1s
800:	learn: 0.9989343	total: 7.41s	remaining: 11.1s
1000:	learn: 1.0000000	total: 9.21s	remaining: 9.19s
1200:	learn: 1.0000000	total: 11.8s	remaining: 7.86s
1400:	learn: 1.0000000	total: 14.1s	remaining: 6.03s
1600:	learn: 1.0000000	total: 15.9s	remaining: 3.96s
1800:	learn: 1.0000000	total: 17.8s	remaining: 1.97s
1999:	learn: 1.0000000	total: 20.5s	remaining: 0us


CatBoostClassifier(bagging_temperature=0.7, border_count=128, class_weights=[1, 19], depth=5, eval_metric='F1', iterations=2000, l2_leaf_reg=7, learning_rate=0.02, loss_function='Logloss', random_seed=42, verbose=200)

In [13]:

X_test = []
for x in splits_names :
  dta = pd.read_csv(x + '/test_full_lightcurves.csv')
  split_map[x] = dta
test_data_log_np = pd.read_csv('test_log.csv')
test_data_log_np = test_data_log_np.drop(
    columns=['Z_err', 'SpecType', 'English Translation']
)
test_data_log_np = test_data_log_np.to_numpy()
for row in test_data_log_np:
    df_split = split_map[row[3]]
    df_obj = df_split[df_split['object_id']==row[0]]
    feats = extract_features(df_obj)
    X_test.append(feats)

X_test = np.array(X_test)
X_test_imp = imputer.transform(X_test)

proba_xgb = xgb_model.predict_proba(X_test_imp)[:,1]
proba_cat = cat_model.predict_proba(X_test_imp)[:,1]
proba_ens = 0.5*proba_xgb + 0.5*proba_cat

y_test_pred = (proba_ens > best_thresh).astype(int)



In [14]:
print(sum(y_test_pred))

578


In [15]:
import numpy as np
import pandas as pd
for x in splits_names :
  dta = pd.read_csv(x + '/train_full_lightcurves.csv')
  split_map[x] = dta
filters = ['u','g','r','i','z','y']
cols_to_stats = ['Flux', 'Flux_err', 'Time (MJD)']
stats_funcs = ['max', 'min', 'mean', 'std', 'sum']

data_train = []

for x in data_log_np:
    object_id = x[0]
    feat_row = [x[1], x[2]]

    df_split = split_map[x[3]]
    df_obj = df_split[df_split['object_id'] == object_id]

    for f in filters:
        df_f = df_obj[df_obj['Filter'] == f]

        for col in cols_to_stats:
            values = df_f[col].values

            if values.size == 0:
                feat_row.extend([0]*6)
            else:
                feat_row.extend([
                    values.max(),
                    values.min(),
                    values.mean(),
                    values.std(),
                    values.sum(),
                    values.size
                ])

    label = x[-1]
    feat_row.append(label)
    data_train.append(feat_row)

data_train = np.array(data_train)


In [16]:
X = data_train[:, :-1]
y = data_train[:, -1].astype(int)


In [17]:
from collections import Counter

counter = Counter(y)
neg, pos = counter[0], counter[1]
scale_pos_weight = neg / pos

print("scale_pos_weight =", scale_pos_weight)


scale_pos_weight = 19.56081081081081


In [39]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=1200,
    depth=7,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="F1",
    class_weights=[1, scale_pos_weight],
    random_seed=42,
    verbose=200
)

model.fit(X, y)


0:	learn: 0.7381610	total: 49.8ms	remaining: 59.7s
200:	learn: 0.9948454	total: 9.03s	remaining: 44.9s
400:	learn: 1.0000000	total: 17s	remaining: 33.8s
600:	learn: 1.0000000	total: 25.8s	remaining: 25.7s
800:	learn: 1.0000000	total: 34.7s	remaining: 17.3s
1000:	learn: 1.0000000	total: 43s	remaining: 8.54s
1199:	learn: 1.0000000	total: 1m 2s	remaining: 0us


CatBoostClassifier(class_weights=[1, 19.56081081081081], depth=7, eval_metric='F1', iterations=1200, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=200)

In [40]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]
model.fit(X_val,y_val)
print(classification_report(y_val, y_pred))
print("AUC =", roc_auc_score(y_val, y_proba))


0:	learn: 0.7311128	total: 110ms	remaining: 2m 11s
200:	learn: 1.0000000	total: 9.15s	remaining: 45.5s
400:	learn: 1.0000000	total: 17.4s	remaining: 34.7s
600:	learn: 1.0000000	total: 24.6s	remaining: 24.6s
800:	learn: 1.0000000	total: 33s	remaining: 16.4s
1000:	learn: 1.0000000	total: 40.3s	remaining: 8.01s


KeyboardInterrupt: 

In [21]:
import numpy as np

thresholds = np.linspace(0.07, 0.08, 10000)
f1_scores = []

for t in thresholds:
    preds = (y_proba > t).astype(int)
    f1_scores.append(f1_score(y_val, preds))

best_t = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print("Best threshold:", best_t)
print("Best F1:", best_f1)


Best threshold: 0.07
Best F1: 1.0


In [41]:
data_log_np = pd.read_csv('test_log.csv')
data_log_np = data_log_np.drop(
    columns=['Z_err', 'SpecType', 'English Translation']
)
data_log_np = data_log_np.to_numpy()

In [23]:
import numpy as np
import pandas as pd
for x in splits_names :
  dta = pd.read_csv(x + '/test_full_lightcurves.csv')
  split_map[x] = dta
filters = ['u','g','r','i','z','y']
cols_to_stats = ['Flux', 'Flux_err', 'Time (MJD)']
stats_funcs = ['max', 'min', 'mean', 'std', 'sum']

data_test = []

for x in data_log_np:
    object_id = x[0]
    feat_row = [x[1], x[2]]

    df_split = split_map[x[3]]
    df_obj = df_split[df_split['object_id'] == object_id]

    for f in filters:
        df_f = df_obj[df_obj['Filter'] == f]

        for col in cols_to_stats:
            values = df_f[col].values

            if values.size == 0:
                feat_row.extend([0]*6)
            else:
                feat_row.extend([
                    values.max(),
                    values.min(),
                    values.mean(),
                    values.std(),
                    values.sum(),
                    values.size
                ])
    data_test.append(feat_row)

data_test = np.array(data_test)


In [42]:
y_pred_test = model.predict_proba(data_test)[:, 1]

In [43]:
print(y_pred_test)

[1.21994702e-03 7.88233674e-04 7.00588259e-03 ... 1.68446924e-03
 8.40837398e-05 4.45065344e-03]


In [44]:
THRESHOLD = 0.032003200320032
y_test_pred = (y_pred_test > THRESHOLD).astype(int)


In [45]:
print(len(y_test_pred))

7135


In [46]:
print(sum(y_test_pred))

901


In [47]:
test_ids_df = pd.read_csv("test_log.csv")
object_id = test_ids_df.iloc[:, 0]

In [48]:

submission = pd.DataFrame({
    "object_id": object_id,
    "prediction": y_test_pred
})


In [49]:
submission.to_csv("submission4.csv", index=False)
print("Fichier submission.csv généré !")

Fichier submission.csv généré !


In [37]:
print(sum(y_test_pred))

3630
